## EDA and Data Inspection on Taiwan Dataset

### Load and explore Taiwan dataset

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# Load Taiwan UCI credit card default dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
df = pd.read_excel(url, header=1, index_col=0)  # Skip first row, use first column as index

In [11]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

# Target variable: default payment next month (1 = default, 0 = no default)
print("\nDefault rate:")
print(df['default payment next month'].value_counts(normalize=True).round(4))

# Numeric summary for key columns
print("\nSummary stats (numeric columns):")
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(df[numeric_cols].describe())

# Correlation with target (top 10)
corr_with_target = df[numeric_cols].corr()['default payment next month'].abs().sort_values(ascending=False)
print("\nTop 10 correlations with default (absolute):")
print(corr_with_target.head(10))


Dataset shape: (30000, 24)

Data types:
LIMIT_BAL                     int64
SEX                           int64
EDUCATION                     int64
MARRIAGE                      int64
AGE                           int64
PAY_0                         int64
PAY_2                         int64
PAY_3                         int64
PAY_4                         int64
PAY_5                         int64
PAY_6                         int64
BILL_AMT1                     int64
BILL_AMT2                     int64
BILL_AMT3                     int64
BILL_AMT4                     int64
BILL_AMT5                     int64
BILL_AMT6                     int64
PAY_AMT1                      int64
PAY_AMT2                      int64
PAY_AMT3                      int64
PAY_AMT4                      int64
PAY_AMT5                      int64
PAY_AMT6                      int64
default payment next month    int64
dtype: object

First 5 rows:
    LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  

**Interpretation:**

The default rate is at roughly 22%, which shows that it's imbalanced but not at an extreme.

Initial insights:
1. We can focus on the F1 or the AUC, and not just merely on the accuracy.
2. The strongest correlations for defaults are the recent payment status variables (such as pay 0 and pay 2, through to pay 6).
3. These are followed by the credit limit and first payment amounts.

Overall, initial data observation shows it matches the intuition that recent delinquency drives default risks.

### Preprocessing + Phase I Proof-of-Concept model 

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [15]:
df_ = df.rename(columns={'default payment next month': 'default'})

X = df_.drop(columns=['default'])
y = df_['default']

# Train/test split (stratified to preserve default rate)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train default rate:", y_train.mean().round(4))
print("Test default rate:", y_test.mean().round(4))

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train shape: (24000, 23)
Test shape: (6000, 23)
Train default rate: 0.2212
Test default rate: 0.2212


### Baseline logistic regression

In [16]:
log_clf = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight=None
)

log_clf.fit(X_train_scaled, y_train)

y_pred = log_clf.predict(X_test_scaled)
y_proba = log_clf.predict_proba(X_test_scaled)[:, 1]

print("Test AUC:", roc_auc_score(y_test, y_proba).round(4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

Test AUC: 0.7076

Classification report:
              precision    recall  f1-score   support

           0     0.8178    0.9692    0.8871      4673
           1     0.6883    0.2396    0.3555      1327

    accuracy                         0.8078      6000
   macro avg     0.7531    0.6044    0.6213      6000
weighted avg     0.7892    0.8078    0.7695      6000



**Interpretation from baseline Logistic Regression Interpretation**

For this first pass, the logistic regression model is performing noticeably better than random with an AUC of roughly 0.71, though there is clearly significant room for improvement. 

The model is very good at identifying non-defaulters (Class 0), with a high precision of 0.82 and nearly perfect recall at 0.97. However, it struggles with the minority class (Class 1). While the precision for defaults is okay at 0.69, meaning when it predicts a default, it’s usually correct, the recall is quite low at 0.24, which tells us the model is missing the vast majority of actual defaulters.

It’s important to note that the overall accuracy of 0.81 is actually quite misleading here due to the class imbalance; the high score is mostly driven by the model's success with non-defaulters while defaults remain under-detected. This really highlights that while a conventional logistic model is a decent starting point, we are being too conservative on defaults. This motivates us to explore ensemble or Bayesian methods in the next steps to specifically improve our recall and F1-score for the default class.